# 10.8.视觉 Transformer（Vision Transformer）

Transformer 架构最初是为序列到序列学习提出的，主要关注机器翻译。随后，Transformer 成为各种自然语言处理任务的首选模型 ([Radford et al., 2018](https://d2l.ai/chapter_references/zreferences.html#Radford.Narasimhan.Salimans.ea.2018), [Radford et al., 2019](https://d2l.ai/chapter_references/zreferences.html#Radford.Wu.Child.ea.2019), [Brown et al., 2020](https://d2l.ai/chapter_references/zreferences.html#brown2020language), [Devlin et al., 2018](https://d2l.ai/chapter_references/zreferences.html#Devlin.Chang.Lee.ea.2018), [Raffel et al., 2020](https://d2l.ai/chapter_references/zreferences.html#raffel2020exploring))。然而，在计算机视觉领域，占主导地位的架构一直是 CNN（见第7章 现代卷积神经网络）。自然，研究人员开始思考，是否可以通过将 Transformer 模型应用到图像数据上来做得更好。这个问题在计算机视觉社区引起了极大的兴趣。最近，Ramachandran 等人提出了用自注意力替代卷积的方案 ([Ramachandran et al., 2019](https://d2l.ai/chapter_references/zreferences.html#ramachandran2019stand))。然而，其在注意力中使用的特殊模式使其难以在硬件加速器上扩展模型。随后，Cordonnier 等人从理论上证明了自注意力可以学会表现得类似于卷积 ([Cordonnier et al., 2020](https://d2l.ai/chapter_references/zreferences.html#cordonnier2020relationship))。经验上，$2 \times 2$ 的图像块被作为输入，但小的图像块尺寸使模型只适用于低分辨率图像数据。

在没有对图像块尺寸施加特定约束的情况下，*视觉 Transformer*（vision Transformer，ViT）从图像中提取图像块，并将它们输入到 Transformer 编码器中，以获得全局表示，最终将其转换为分类结果 ([Dosovitskiy et al., 2021](https://d2l.ai/chapter_references/zreferences.html#Dosovitskiy.Beyer.Kolesnikov.ea.2021))。值得注意的是，Transformer 表现出比 CNN 更好的可扩展性：当在更大的数据集上训练更大的模型时，视觉 Transformer 以显著的优势优于 ResNet。与自然语言处理中网络架构设计的格局类似，Transformer 也已成为计算机视觉领域的游戏规则改变者。

## 10.8.1.环境配置


In [ ]:
%pip install pypto==0.2.1 torch torch_npu matplotlib torchvision

import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import pypto
import torch
from torch import nn
import torch_npu
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

import math

from src.pypto_ops import (PyPTOLinear, PyPTOSigmoid, PyPTOMul,
                           PyPTOConv2d, PyPTOLayerNormModule, loss_fn)
from src.attention import (masked_softmax, DotProductAttention,
                           transpose_qkv, transpose_output, MultiHeadAttention)
from src.utils import _accuracy, Animator

torch.manual_seed(0)


## 10.8.2.模型

<div align="center">
  <img src="./images/vit.svg" alt="图10.8.1 视觉Transformer架构" width="400">
  <br><small>图10.8.1（英文版 Fig. 11.8.1）视觉 Transformer 架构</small>
</div> 描绘了视觉 Transformer 的模型架构。该架构由一个将图像切分为图像块的 stem、一个基于多层 Transformer 编码器的主体，以及一个将全局表示转换为输出标签的 head 组成。

考虑一个高度为$h$、宽度为$w$、通道数为$c$的输入图像。指定图像块的高度和宽度均为$p$，将图像切分为$m = hw/p^2$个图像块的序列，其中每个图像块被展平为长度为$cp^2$的向量。通过这种方式，图像块可以被 Transformer 编码器视为文本序列中的词元。一个特殊的“&lt;cls&gt;”（类别）标记和$m$个展平的图像块被线性投影为$m+1$个向量的序列，并与可学习的*位置编码*相加。多层 Transformer 编码器将$m+1$个输入向量转换为相同数量的等长输出向量表示。其工作方式与 图10.7.1中的原始 Transformer 编码器完全相同，只是在规范化的位置有所不同。由于“&lt;cls&gt;”标记通过自注意力关注所有图像块（参见图10.6.1），其来自 Transformer 编码器输出的表示将被进一步转换为输出标签。

## 10.8.3.图像块嵌入

为了实现视觉 Transformer，让我们从 图10.8.1 中的图像块嵌入开始。将图像切分为图像块，并对这些展平的图像块进行线性投影，可以简化为单个卷积操作，其中卷积核大小和步幅大小都设置为图像块大小。


In [2]:
class PatchEmbedding(nn.Module):
    """图像块嵌入"""
    def __init__(self, img_size=96, patch_size=16, num_hiddens=512,
                 in_channels=None):
        super().__init__()
        def _make_tuple(x):
            if not isinstance(x, (list, tuple)):
                return (x, x)
            return x
        img_size, patch_size = _make_tuple(img_size), _make_tuple(patch_size)
        self.img_size, self.patch_size = img_size, patch_size
        self.num_patches = (img_size[0] // patch_size[0]) * (
            img_size[1] // patch_size[1])
        self.num_hiddens = num_hiddens
        # 通道数由 in_channels 指定或首次 forward 按输入动态确定（不硬编码 3）
        self.conv = None
        if in_channels is not None:
            self.conv = PyPTOConv2d(
                in_channels, num_hiddens, patch_size, stride=patch_size)

    def forward(self, X):
        # X: (batch_size, channels, img_h, img_w)
        if self.conv is None:
            self.conv = PyPTOConv2d(
                X.shape[1], self.num_hiddens, self.patch_size,
                stride=self.patch_size).to(X.device)
        return self.conv(X).flatten(2).transpose(1, 2)


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>通道补零</b>：NPU 数据在核内按 <code>NC1HWC0</code> 格式存储，C0 是通道维的最小对齐粒度，<code>pypto.conv</code> 要求输入通道数为 C0 的倍数（FP32 为 8，FP16/BF16 为 16），Fashion-MNIST 为单通道灰度图，故 <code>PyPTOConv2d</code> 自动把通道补零到 8（补零通道权重亦为 0、不参与计算，数学上严格等价）。</li>
      <li style="margin: 0 0 8px 0;"><b>tile 约束</b>：AI Core 的片上存储分 L1 → L0（Cube 输入缓冲）→ UB（向量工作区）三级，数据逐层搬运需按块（tile）切分，conv 用 <code>set_conv_tile_shapes</code> 配置 L1/L0 tile、<code>set_vec_tile_shapes</code> 配置 UB 上的 vec tile。96×96 图像切 16×16 块后输出为 6×6（宽非 16 倍数），tile 取 <code>tileHout=1, tileWout=16</code>；vec tile 首维需为 16 倍数、末维 32 字节对齐、H 维取小值以控制 transdata（NCHW-NC1HWC0 格式转换，在 UB 中进行）的 workspace。上述推导封装在 <code>PyPTOConv2d._build_impl</code> 中。</li>
      <li style="margin: 0 0 8px 0;"><b>反向</b>：pypto 暂无 conv 反向 kernel，<code>PyPTOConv2d</code> 用卷积梯度的等价公式（<code>grad_w = unfold(x)^T @ grad_out</code>、<code>grad_x = fold(weight^T @ grad_out)</code>，torch 实现）反向，梯度与 <code>torch.nn.functional.conv2d</code> 逐项对比一致（误差 &lt; 1e-3 量级）。</li>
    </ul>
  </div>
</details>
<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=96, patch_size=16, num_hiddens=512):
        super().__init__()
        def _make_tuple(x):
            if not isinstance(x, (list, tuple)):
                return (x, x)
            return x
        img_size, patch_size = _make_tuple(img_size), _make_tuple(patch_size)
        self.num_patches = (img_size[0] // patch_size[0]) * (
            img_size[1] // patch_size[1])
        self.conv = nn.LazyConv2d(num_hiddens, kernel_size=patch_size,
                                  stride=patch_size)
    def forward(self, X):
        # Output shape: (batch size, no. of patches, no. of channels)
        return self.conv(X).flatten(2).transpose(1, 2)
    </pre>
  </div>
</details>

在下面的示例中，将高度和宽度为`img_size`的图像作为输入，图像块嵌入输出`(img_size//patch_size)**2`个图像块，这些图像块被线性投影为长度为`num_hiddens`的向量。


In [3]:
img_size, patch_size, num_hiddens, batch_size = 96, 16, 512, 4
patch_emb = PatchEmbedding(img_size, patch_size, num_hiddens,
                           in_channels=1).to(device)
X = torch.zeros(batch_size, 1, img_size, img_size, device=device)
print('输出形状：', tuple(patch_emb(X).shape), '，预期：',
      (batch_size, (img_size // patch_size) ** 2, num_hiddens))


输出形状： (4, 36, 512) ，预期： (4, 36, 512)


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
img_size, patch_size, num_hiddens, batch_size = 96, 16, 512, 4
patch_emb = PatchEmbedding(img_size, patch_size, num_hiddens)
X = torch.zeros(batch_size, 3, img_size, img_size)
d2l.check_shape(patch_emb(X),
                (batch_size, (img_size//patch_size)**2, num_hiddens))
    </pre>
  </div>
</details>

## 10.8.4.视觉 Transformer 编码器

视觉 Transformer 编码器的多层感知机与原始 Transformer 编码器的基于位置的前馈网络略有不同（参见10.7节中的基于位置的前馈网络）。首先，这里的激活函数使用高斯误差线性单元（GELU），可以看作是 ReLU 的平滑版本 ([Hendrycks and Gimpel, 2016](https://d2l.ai/chapter_references/zreferences.html#Hendrycks.Gimpel.2016))。其次，为了正则化，对多层感知机中每个全连接层的输出应用暂退法（dropout）。

由于 pypto 暂无精确 `erf` 实现，本节用 $x\cdot\sigma(1.702x)$ 近似标准 GELU（两者最大绝对误差约 $10^{-3}$ 量级，与 `torch.nn.GELU` 数值行为略有差异，对训练与推理结果影响可忽略）。

In [4]:
class PyPTOGELU(nn.Module):

    def forward(self, x):
        return PyPTOMul.apply(x, PyPTOSigmoid.apply(x * 1.702))


class ViTMLP(nn.Module):
    """视觉 Transformer 的多层感知机（PyPTO 算子版）"""

    def __init__(self, mlp_num_input, mlp_num_hiddens, mlp_num_outputs,
                 dropout=0.5):
        super().__init__()
        self.dense1 = PyPTOLinear(mlp_num_input, mlp_num_hiddens)
        self.gelu = PyPTOGELU()
        self.dropout1 = nn.Dropout(dropout)
        self.dense2 = PyPTOLinear(mlp_num_hiddens, mlp_num_outputs)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(
            self.dense1(x)))))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class ViTMLP(nn.Module):
    def __init__(self, mlp_num_hiddens, mlp_num_outputs, dropout=0.5):
        super().__init__()
        self.dense1 = nn.LazyLinear(mlp_num_hiddens)
        self.gelu = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)
        self.dense2 = nn.LazyLinear(mlp_num_outputs)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(
            self.dense1(x)))))
    </pre>
  </div>
</details>

视觉 Transformer 编码器块的实现遵循 图10.8.1 中的预规范化设计，其中规范化正好应用在多头注意力或多层感知机*之前*。与后规范化（图10.7.1中的 "add & norm"）相比，后规范化将规范化放在残差连接*之后*，预规范化可以使 Transformer 的训练更有效或更高效 ([Baevski and Auli, 2018](https://d2l.ai/chapter_references/zreferences.html#baevski2018adaptive), [Wang et al., 2019](https://d2l.ai/chapter_references/zreferences.html#wang2019learning), [Xiong et al., 2020](https://d2l.ai/chapter_references/zreferences.html#xiong2020layer))。


In [5]:
class ViTBlock(nn.Module):
    """视觉 Transformer 编码器块（预规范化 + 残差连接，PyPTO 算子版）"""

    def __init__(self, num_hiddens, norm_shape, mlp_num_hiddens,
                 num_heads, dropout, use_bias=False):
        super().__init__()
        self.ln1 = PyPTOLayerNormModule(norm_shape)
        self.attention = MultiHeadAttention(
            num_hiddens, num_hiddens, num_hiddens, num_hiddens, num_heads,
            dropout, use_bias)
        self.ln2 = PyPTOLayerNormModule(norm_shape)
        self.mlp = ViTMLP(num_hiddens, mlp_num_hiddens, num_hiddens, dropout)

    def forward(self, X, valid_lens=None):
        # 预规范化：先 LayerNorm 再进入注意力 / MLP
        X = X + self.attention(*([self.ln1(X)] * 3), valid_lens)
        return X + self.mlp(self.ln2(X))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class ViTBlock(nn.Module):
    def __init__(self, num_hiddens, norm_shape, mlp_num_hiddens,
                 num_heads, dropout, use_bias=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(norm_shape)
        self.attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                dropout, use_bias)
        self.ln2 = nn.LayerNorm(norm_shape)
        self.mlp = ViTMLP(mlp_num_hiddens, num_hiddens, dropout)
    def forward(self, X, valid_lens=None):
        X = X + self.attention(*([self.ln1(X)] * 3), valid_lens)
        return X + self.mlp(self.ln2(X))
    </pre>
  </div>
</details>

正如10.7节中的Transformer编码器那样，视觉 Transformer 编码器块不会改变其输入形状。


In [6]:
X = torch.ones((2, 100, 24), device=device)
encoder_blk = ViTBlock(24, 24, 48, 8, 0.5).to(device)
encoder_blk.eval()
print('输出形状：', tuple(encoder_blk(X).shape), '，与输入一致')

输出形状： (2, 100, 24) ，与输入一致


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
X = torch.ones((2, 100, 24))
encoder_blk = ViTBlock(24, 24, 48, 8, 0.5)
encoder_blk.eval()
d2l.check_shape(encoder_blk(X), X.shape)
    </pre>
  </div>
</details>

## 10.8.5.整体实现

下面视觉 Transformer 的前向过程很简单。首先，将输入图像送入`PatchEmbedding`实例，其输出与“&lt;cls&gt;”标记嵌入拼接。在暂退法之前，它们与可学习的*位置编码*相加。然后将输出送入堆叠了`num_blks`个`ViTBlock`类实例的 Transformer 编码器。最后，网络头部将“&lt;cls&gt;”标记的表示投影到输出。


In [7]:
class ViT(nn.Module):
    """视觉 Transformer（PyPTO 算子版）"""

    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout,
                 num_classes=10, in_channels=3):
        super().__init__()
        self.patch_embedding = PatchEmbedding(
            img_size, patch_size, num_hiddens, in_channels=in_channels)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))
        num_steps = self.patch_embedding.num_patches + 1  # 加上 cls 标记
        # 位置编码可学习
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hiddens))
        self.dropout = nn.Dropout(emb_dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout))
        self.head = nn.Sequential(PyPTOLayerNormModule(num_hiddens),
                                  PyPTOLinear(num_hiddens, num_classes))

    def forward(self, X):
        X = self.patch_embedding(X)
        X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)
        X = self.dropout(X + self.pos_embedding)
        for blk in self.blks:
            X = blk(X)
        # 取 cls 标记的表示做分类
        return self.head(X[:, 0])

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class ViT(d2l.Classifier):
    &quot;&quot;&quot;Vision Transformer.&quot;&quot;&quot;
    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout, lr=0.1,
                 use_bias=False, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.patch_embedding = PatchEmbedding(
            img_size, patch_size, num_hiddens)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))
        num_steps = self.patch_embedding.num_patches + 1  # Add the cls token
        # Positional embeddings are learnable
        self.pos_embedding = nn.Parameter(
            torch.randn(1, num_steps, num_hiddens))
        self.dropout = nn.Dropout(emb_dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f&quot;{i}&quot;, ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout, use_bias))
        self.head = nn.Sequential(nn.LayerNorm(num_hiddens),
                                  nn.Linear(num_hiddens, num_classes))
    def forward(self, X):
        X = self.patch_embedding(X)
        X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)
        X = self.dropout(X + self.pos_embedding)
        for blk in self.blks:
            X = blk(X)
        return self.head(X[:, 0])
    </pre>
  </div>
</details>

## 10.8.6.训练

在 Fashion-MNIST 数据集上训练视觉 Transformer 与在第 7 章现代卷积神经网络中训练 CNN 的方式相同。

**编译提示**：训练 cell 首次运行会触发大量 PyPTO kernel 的编译，编译耗时较长，请耐心等待；编译产物不跨进程复用，重启 notebook 内核后需重新编译。


In [8]:
import time
import torchvision
from torchvision import transforms

img_size, patch_size = 96, 16
num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 2
emb_dropout, blk_dropout, lr = 0.1, 0.1, 0.1
num_epochs, batch_size = 10, 128

trans = transforms.Compose([transforms.Resize((img_size, img_size)),
                            transforms.ToTensor()])
mnist_train = torchvision.datasets.FashionMNIST(
    root="../data", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, transform=trans, download=True)
train_iter = torch.utils.data.DataLoader(mnist_train, batch_size, shuffle=True,
                                         num_workers=4)
test_iter = torch.utils.data.DataLoader(mnist_test, batch_size, shuffle=False,
                                        num_workers=4)

In [9]:
model = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
            num_blks, emb_dropout, blk_dropout,
            in_channels=1).to(device)  # FashionMNIST 为单通道


def init_weights(m):
    """用 kaiming 均匀初始化替代默认小尺度初始化，与 nn.Linear 默认一致"""
    if isinstance(m, (PyPTOLinear, PyPTOConv2d)):
        nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
        if m.bias is not None:
            nn.init.zeros_(m.bias)


model.apply(init_weights)
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

In [10]:
def evaluate_accuracy(net, data_iter, device):
    """在测试集上评估准确率"""
    net.eval()
    metric = [0.0, 0.0]  # 正确数, 样本数
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            metric[0] += _accuracy(net(X), y)
            metric[1] += y.numel()
    return metric[0] / metric[1]


# 与原书 `d2l.Trainer.fit` 一致：每轮记录 loss / train acc / test acc 并绘制曲线
animator = Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0, 1],
                    legend=['loss', 'train acc', 'test acc'])

timer = time.time()
for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc = 0.0, 0.0
    n = 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        y_hat = model(X)
        l = loss_fn(y_hat, y, num_classes=10)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        train_loss += l.item() * y.numel()
        train_acc += _accuracy(y_hat, y)
        n += y.numel()
    test_acc = evaluate_accuracy(model, test_iter, device)
    animator.add(epoch + 1, (train_loss / n, train_acc / n, test_acc))
    print(f'epoch {epoch + 1}, loss {train_loss / n:.3f}, '
          f'train acc {train_acc / n:.3f}, test acc {test_acc:.3f}')
print(f'总训练耗时 {time.time() - timer:.1f} 秒')

epoch 10, loss 0.390, train acc 0.856, test acc 0.863
总训练耗时 1632.0 秒


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
img_size, patch_size = 96, 16
num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 2
emb_dropout, blk_dropout, lr = 0.1, 0.1, 0.1
model = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
            num_blks, emb_dropout, blk_dropout, lr)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128, resize=(img_size, img_size))
trainer.fit(model, data)
    </pre>
  </div>
</details>

## 10.8.7.小结

你可能已经注意到，对于 Fashion-MNIST 这样的小数据集，我们实现的视觉 Transformer 并不优于残差网络（ResNet）一节。即使在 ImageNet 数据集（120 万张图像）上也可以观察到类似的情况。这是因为 Transformer *缺乏*卷积中的那些有用原理，例如平移不变性和局部性（见从全连接层到卷积一节）。然而，当在更大的数据集（例如 3 亿张图像）上训练更大的模型时，情况发生了变化：视觉 Transformer 在图像分类中以较大优势优于 ResNet，这证明了 Transformer 在可扩展性方面的内在优势 ([Dosovitskiy et al., 2021](https://d2l.ai/chapter_references/zreferences.html#Dosovitskiy.Beyer.Kolesnikov.ea.2021))。视觉 Transformer 的引入改变了建模图像数据的网络设计格局。它们很快被证明在 ImageNet 数据集上通过 DeiT 的数据高效训练策略是有效的 ([Touvron et al., 2021](https://d2l.ai/chapter_references/zreferences.html#touvron2021training))。然而，自注意力的二次复杂度（[10.6节](10.06_self_attention_and_positional_encoding.ipynb)）使 Transformer 架构不太适合更高分辨率的图像。为了构建计算机视觉中的通用主干网络，Swin Transformer 解决了与图像大小相关的二次计算复杂度（10.6节中的自注意力与卷积/循环神经网络的比较），并恢复了类似卷积的先验，将 Transformer 的应用扩展到图像分类之外的一系列计算机视觉任务，并取得了最先进的结果 ([Liu et al., 2021](https://d2l.ai/chapter_references/zreferences.html#liu2021swin))。


## 10.8.8.练习

1. `img_size`的值如何影响训练时间？
1. 如果不将`"<cls>"`标记的表示投影到输出，而是投影图像块表示的均值，你会怎么做？实现这一更改，看看它如何影响准确率。
1. 你能修改超参数来提高视觉 Transformer 的准确率吗？

参考答案详见 [answers/10.08_reference_answer](./answers/10.08_reference_answer.ipynb)。


### 10.8.8.1.参考答案


In [ ]:
!cat answers/txt/10.08_reference_answer.txt